In [1]:
!pip install langchain
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.3 MB/s eta 0:00:00


In [1]:
!pip install -U langchain
!pip install -U langchain-core
!pip install -U langchain-community
!pip install -U langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from langchain_core.prompts import PromptTemplate

from langchain_core.output_parsers import JsonOutputParser

from langchain_groq import ChatGroq

In [3]:
llm = ChatGroq(
    groq_api_key="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",
    model_name="llama-3.3-70b-versatile"
)

In [18]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(

    input_variables=[
        "job_description",
        "resume_text"
    ],

    template="""
You are an advanced AI Resume Screening Assistant.

Your task is to carefully compare the candidate resume with the job description and generate a professional evaluation.

You must:
- deeply analyze technical skills
- compare experience relevance
- identify matched and missing skills
- infer related technologies when appropriate
- generate a realistic score

IMPORTANT SCORING RULES:
- 80-100 → Strong Fit
- 60-79 → Potential Fit
- Below 60 → Not a Fit

SEMANTIC MATCHING RULES:
- SQLite counts as partial SQL experience
- FastAPI counts similar to Flask
- React Native counts partially for React
- Similar frameworks should be considered partial matches

STRICT OUTPUT RULES:
- Return ONLY valid JSON
- Do NOT return markdown
- Do NOT return explanation outside JSON
- Do NOT add extra text

OUTPUT JSON FORMAT:
{{
    "match_score": 0,
    "matched_skills": [],
    "missing_skills": [],
    "summary": "",
    "recommendation": ""
}}

FIELD RULES:
- match_score must be integer between 0 and 100
- matched_skills must contain only matched technical skills
- missing_skills must contain only important missing skills
- summary should be 2-3 professional lines
- recommendation must be exactly one of:
    - Strong Fit
    - Potential Fit
    - Not a Fit

JOB DESCRIPTION:
{job_description}

RESUME:
{resume_text}
"""
)

In [21]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

chain = prompt | llm | parser

In [24]:
# =====================================
# AI RESUME SCREENING PIPELINE
# =====================================

# =====================================
# IMPORT REQUIRED LIBRARIES
# =====================================

# !pip install pdfplumber

import json
import pdfplumber

from google.colab import files


# =====================================
# STEP 1 — GET JOB DESCRIPTION
# =====================================

print("=====================================")
print(" ENTER JOB DESCRIPTION ")
print("=====================================\n")

job_description = input("Paste Job Description Here:\n")


# =====================================
# STEP 2 — UPLOAD RESUME PDF
# =====================================

print("\n=====================================")
print(" UPLOAD RESUME PDF ")
print("=====================================\n")

uploaded = files.upload()

uploaded_file_name = list(uploaded.keys())[0]


# =====================================
# STEP 3 — EXTRACT TEXT FROM PDF
# =====================================

resume_text = ""

try:

    print("\nExtracting Resume Text...\n")

    with pdfplumber.open(uploaded_file_name) as pdf:

        for page in pdf.pages:

            text = page.extract_text()

            if text:
                resume_text += text + "\n"

    # =====================================
    # VALIDATION CHECK
    # =====================================

    if len(resume_text.strip()) < 50:
        raise Exception(
            "Resume text is too short or unreadable."
        )

    print("Resume Extracted Successfully")
    print(
        f"Total Extracted Characters: {len(resume_text)}"
    )

except Exception as e:

    print("\nPDF Extraction Failed")
    print(f"Error: {str(e)}")


# =====================================
# STEP 4 — RUN LANGCHAIN PIPELINE
# =====================================

try:

    print("\n=====================================")
    print(" RUNNING AI RESUME ANALYSIS ")
    print("=====================================\n")

    result = chain.invoke(
        {
            "job_description": job_description,
            "resume_text": resume_text
        }
    )

    print("AI Analysis Completed Successfully")


except Exception as e:

    print("\nLangChain Pipeline Failed")
    print(f"Error: {str(e)}")


# =====================================
# STEP 5 — DISPLAY FINAL RESULT
# =====================================

try:

    print("\n=====================================")
    print(" FINAL RESUME EVALUATION ")
    print("=====================================\n")

    print(
        json.dumps(
            result,
            indent=4
        )
    )

    # =====================================
    # OPTIONAL MATCH SCORE DISPLAY
    # =====================================

    if "match_score" in result:

        print(
            f"\nFinal Match Score: "
            f"{result['match_score']}/100"
        )

    # =====================================
    # OPTIONAL CAREER SUGGESTIONS
    # =====================================

    if (
        "match_score" in result and
        result["match_score"] < 70
    ):

        print("\n=====================================")
        print(" CAREER SUGGESTIONS ")
        print("=====================================\n")

        print("1. LinkedIn Jobs")
        print("   https://www.linkedin.com/jobs/\n")

        print("2. Unstop")
        print("   https://unstop.com/jobs\n")

        print("3. Internshala")
        print("   https://internshala.com/\n")


except Exception as e:

    print("\nResult Display Failed")
    print(f"Error: {str(e)}")

 ENTER JOB DESCRIPTION 

Paste Job Description Here:
We are hiring a Cybersecurity Analyst.  Required Skills: •⁠  ⁠Network Security •⁠  ⁠Linux •⁠  ⁠Penetration Testing •⁠  ⁠Security Tools  Preferred Skills: •⁠  ⁠Python •⁠  ⁠SIEM Tools •⁠  ⁠Ethical Hacking  Experience Required: 1-3 years  Responsibilities: •⁠  ⁠Monitor system security •⁠  ⁠Identify vulnerabilities •⁠  ⁠Conduct security audits •⁠  ⁠Respond to cyber threats

 UPLOAD RESUME PDF 



Saving Vraj_Suratwala_Resume_MSCIT.pdf to Vraj_Suratwala_Resume_MSCIT (3).pdf

Extracting Resume Text...

Resume Extracted Successfully
Total Extracted Characters: 5625

 RUNNING AI RESUME ANALYSIS 

AI Analysis Completed Successfully

 FINAL RESUME EVALUATION 

{
    "match_score": 40,
    "matched_skills": [
        "Linux",
        "Python"
    ],
    "missing_skills": [
        "Network Security",
        "Penetration Testing",
        "Security Tools",
        "SIEM Tools",
        "Ethical Hacking"
    ],
    "summary": "The candidate has a strong foundation in AI, IoT, and full-stack development, but lacks direct experience in cybersecurity. However, they have some relevant technical skills such as Linux and Python. The candidate's experience is mostly in development and projects, with no direct experience in cybersecurity analysis.",
    "recommendation": "Not a Fit"
}

Final Match Score: 40/100

 CAREER SUGGESTIONS 

1. LinkedIn Jobs
   https://www.linkedin.com/jobs/

2. Unsto

In [17]:
# Thank you!